# Outlier testing against the existing model

This notebook compares each verified charge against the legacy sentencing model and highlights charges where the gap is large enough to merit manual review.

In [5]:
from __future__ import annotations

from pathlib import Path
from typing import Any

import pandas as pd
from dotenv import load_dotenv

repo_root = Path.cwd().resolve()
if not (repo_root / 'featureExtraction').exists():
    repo_root = repo_root.parent

for env_path in (repo_root / 'featureExtraction' / '.env', repo_root / 'featureVerification' / '.env.local', repo_root / '.env'):
    if env_path.exists():
        load_dotenv(env_path)

from evaluate_verified_sentences import build_model_input, get_collection, get_total_months
from legacy_model import DkPredictor

verified_collection, _ = get_collection()
predictor = DkPredictor()
query = {'is_verified': True}
projection = {'filename': 1, 'judgement.neutral_citation': 1, 'exclude': 1, 'remarks': 1, 'trials': 1}
docs = list(verified_collection.find(query, projection))

OUTLIER_THRESHOLD_MONTHS = 6

def format_drugs(trial: dict[str, Any]) -> str:
    parts = []
    for drug in trial.get('drugs') or []:
        drug_type = drug.get('drug_type')
        quantity = drug.get('quantity')
        if drug_type:
            parts.append(f"{drug_type}:{quantity}")
    return '; '.join(parts)

rows = []
for doc in docs:
    trials = (doc.get('trials') or {}).get('trials') or []
    for index, trial in enumerate(trials):
        model_input = build_model_input(trial)
        predicted_months = int(predictor.explain(model_input)['final_sentence'])
        actual_months = get_total_months(trial.get('final_sentence'))
        difference_months = predicted_months - actual_months
        if abs(difference_months) >= OUTLIER_THRESHOLD_MONTHS:
            rows.append({
                'neutral_citation': (doc.get('judgement') or {}).get('neutral_citation'),
                'exclude_case': bool(doc.get('exclude')),
                'trial_index': index,
                'charge_no': (trial.get('charge_type') or {}).get('charge_no'),
                'charge_name': (trial.get('charge_type') or {}).get('charge_name'),
                'drugs': format_drugs(trial),
                'legacy_model_predicted_months': predicted_months,
                'actual_months': actual_months,
                'difference_months': difference_months,
                'absolute_difference_months': abs(difference_months),
                'remarks': doc.get('remarks'),
            })

outlier_df = pd.DataFrame(rows).sort_values(['difference_months', 'actual_months'], ascending=[False, False]).reset_index(drop=True)
output_dir = repo_root / 'notebooks'
output_dir.mkdir(exist_ok=True)
try:
    outlier_df.to_excel(output_dir / 'model_outlier_review.xlsx', index=False)
except Exception as exc:
    print(f'Excel export skipped: {exc}')

print(f'Found {len(outlier_df)} outlier candidates using a threshold of {OUTLIER_THRESHOLD_MONTHS} months')
outlier_df.head()

Found 1117 outlier candidates using a threshold of 6 months


,neutral_citation,exclude_case,trial_index,charge_no,charge_name,drugs,legacy_model_predicted_months,actual_months,difference_months,absolute_difference_months,remarks
0,[2022] HKCFI 183,True,0,1,Trafficking in a dangerous drug,Cocaine:14300,356,0,356,356,Ms Ramirez found to have no case to answer for...
1,[2023] HKCFI 2256,False,1,2,Conspiracy to traffic in dangerous drugs,Methamphetamine:10860,341,0,341,341,
2,[2021] HKCFI 3788,True,0,1,Trafficking in a dangerous drug,Cocaine:6000,320,0,320,320,not convicted for trafficking in dangerous drgus
3,[2022] HKCFI 3370,True,1,1,Trafficking in a dangerous drug,Methamphetamine:3158.5,301,0,301,301,D2 has not been convicted of drug trafficking.
4,[2024] HKCFI 3175,True,1,2,Trafficking in a dangerous drug,Cocaine:968.61,262,0,262,262,There is only the final total imprisonment sen...
